# imported all libraries

Here I have imported all necessary libraries that will be used for text processing, NLP, and clustering.

It is done for handling text tokenization, named entity recognition, and machine learning-based categorization. 

In [52]:
import nltk
import spacy
import re
import pandas as pd
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import stopwords
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from spacy.matcher import Matcher

Download necessary NLTK resources

In [53]:
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gchaw\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\gchaw\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gchaw\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

------------->

Here I have created a function to preprocess the text.


It is done for converting text to lowercase, removing special characters, tokenizing into sentences and removing stopwords.

In [54]:
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters but keep sentence structure
    text = re.sub(r'[^\w\s\.]', '', text)
    
    # Split into sentences
    sentences = sent_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    processed_sentences = []
    
    for sentence in sentences:
        words = word_tokenize(sentence)
        words = [word for word in words if word not in stop_words]
        processed_sentences.append(' '.join(words))
    
    return processed_sentences

----------------->

Here I have defined a dictionary containing task-related keywords.


It is done for identifying whether a sentence contains a task based on modal verbs, action verbs, and time indicators.

In [55]:
def get_task_patterns():
    return {
        'modal_verbs': {'must', 'should', 'have to', 'need to', 'has to', 'had to'},
        'action_verbs': {'complete', 'finish', 'submit', 'prepare', 'review', 
                        'update', 'create', 'develop', 'implement', 'clean', 
                        'buy', 'get', 'send', 'write', 'call'},
        'temporal_markers': {'by', 'before', 'after', 'when', 'while', 'during', 
                           'until', 'deadline', 'tomorrow', 'today', 'next'}
    }

-------------->

Here I have created a function to extract time-related phrases.


It is done using spaCy’s Named Entity Recognition to detect dates and times from text.

In [61]:
def extract_entities(sentence):
    words = word_tokenize(sentence)
    tagged = pos_tag(words)
    
    person = None
    deadline = None
    
    # Extract person (proper nouns)
    for word, tag in tagged:
        if tag.startswith('NNP'):
            person = word
            break
    
    # Extract deadline
    time_patterns = [
        r'by\s+([\w\s]+)',
        r'before\s+([\w\s]+)',
        r'due\s+([\w\s]+)',
        r'until\s+([\w\s]+)',
        r'on\s+([\w\s]+)'
    ]
    
    for pattern in time_patterns:
        match = re.search(pattern, sentence.lower())
        if match:
            deadline = match.group(1).strip()
            break
            
    return person, deadline

------------->

Here I have created a function to check if a sentence contains a task.


It is done by checking the presence of modal verbs, action verbs, and imperative structures.

In [62]:
def is_task_sentence(sentence, patterns):
    words = word_tokenize(sentence.lower())
    
    # Check for modal verbs and action verbs
    for word in words:
        if word in patterns['modal_verbs'] or word in patterns['action_verbs']:
            return True
    
    # Check for imperative sentences
    tagged = pos_tag(words)
    if tagged and tagged[0][1].startswith('VB'):
        return True
        
    return False

---------->

Here I have implemented a function to categorize extracted tasks.


It is done using TF-IDF vectorization and K-Means clustering to group similar tasks into categories.


In [63]:
def categorize_tasks(tasks, n_categories=5):
    if not tasks:
        return []
        
    # Create TF-IDF matrix
    vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
    tfidf_matrix = vectorizer.fit_transform([task['task'] for task in tasks])
    
    # Perform clustering
    n_categories = min(n_categories, len(tasks))
    kmeans = KMeans(n_clusters=n_categories, random_state=42)
    clusters = kmeans.fit_predict(tfidf_matrix)
    
    # Get category labels based on top terms
    feature_names = vectorizer.get_feature_names_out()
    cluster_centers = kmeans.cluster_centers_
    
    categories = []
    for center in cluster_centers:
        top_terms_idx = center.argsort()[-2:][::-1]
        category = ' & '.join([feature_names[i] for i in top_terms_idx])
        categories.append(category)
    
    return [categories[cluster] for cluster in clusters]

------------>


Here I have created a function to extract tasks, responsible persons, and deadlines from text.


It is done by using previous functions to analyze text and extract relevant details.

In [59]:
def extract_tasks(text):
    try:
        patterns = get_task_patterns()
        sentences = preprocess_text(text)
        
        tasks = []
        for sentence in sentences:
            if is_task_sentence(sentence, patterns):
                person, deadline = extract_entities(sentence)
                task_info = {
                    'task': sentence.strip(),
                    'assignee': person,
                    'deadline': deadline
                }
                tasks.append(task_info)
        
        if tasks:
            df = pd.DataFrame(tasks)
            df['category'] = categorize_tasks(tasks)
            return df
        
        return pd.DataFrame()
    
    except Exception as e:
        print(f"Error in task extraction: {str(e)}")
        return pd.DataFrame()

------------>

Here I have tested the full pipeline with sample text.


It is done to verify if tasks are extracted correctly and structured properly in a DataFrame.

In [64]:
def test_implementation():
    test_text = """
    Rahul wakes up early every day.
    He goes to college in the morning and comes back at 3 pm.
    At present, Rahul is outside. """

    results = extract_tasks(test_text)
    print("\nExtracted Tasks:")
    print(results)
    

# Run the test
if __name__ == "__main__":
    test_implementation()

C:\Users\gchaw\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(



Extracted Tasks:
                                     task assignee deadline      category
0  goes college morning comes back 3 pm .     None     None  pm & morning
